In [128]:
import numpy as np

from matplotlib import pyplot as plt
from scipy import stats

# Importance Sampling

Importance sampling (IS), despite the name, is not a tool for generating samples from a target distribution. 
IS is an algorithm used to approximate the expectation
$$
E_X[g(X)]\;.
$$
More specifically it is an algorithm used for variance reduction of the approximation error of Monte Carlo integration. Via Monte Carlo integration, an expectation $E_X[g(X)]$ where $X \sim f_X(x)$ is normally approximated by independently generating $n$ samples $(x_1, \ldots, x_n)$ from $f_X(x)$ 
and use the sample mean 
$$
\frac{1}{n}\,\sum_{i=1}^n g(x_i)
$$
as an approximation to the integral $E_X[g(X)]$. 

When the regions where $g(x)$ is large align with the regions where $f_X(x)$ has high probability mass, most random draws from $f_X$ yield meaningful, non-zero values for $g(x)$. The sample variance of $g(X)$ remains low, and the sample mean $\frac{1}{n}\sum g(X)$ converges efficiently to the true expected value without extreme fluctuations.

When $g(x)$ has large values exclusively in regions where $f_X(x)$ has near-zero probability mass, the vast majority of draws from $f_X$ will yield $g(x) \approx 0$. Because the true integral relies heavily on those rare instances where $x$ lands in the tail, standard Monte Carlo will usually miss them entirely, outputting an estimate very close to zero. On the rare occasion the simulation does hit that low-probability region, the value of $g(x)$ will be massive, causing a massive spike in the sample mean. While the estimator remains mathematically unbiased as $n \to \infty$, this extreme fluctuation (high variance) means the finite-sample average $\frac{1}{n}\sum g(X)$ is highly unreliable.

Importance sampling can correct for this by defining a new distribution $f_Y$ that places high probability mass exactly where $g(x)$ is large ensuring the simulation spends its computational budget generating samples that actually contribute meaningfully to the integral. In order to preserve the expectation, importance sampling utilizes the following fact:

If $X \in \mathbb{R}^{K}$ a continuous random vector with support $R_X$ and joint PDF $f_X(x)$, $g(x) : \mathbb{R}^K \rightarrow \mathbb{R}$ a function, and $Y \in \mathbb{R}^K$ another continuous random vector with joint PDF $f_Y(y)$ such that $f_Y(x) > 0$ whenever $f_X(x) > 0$, then the following holds
$$
E[g(X)] = E[\frac{f_X(Y)}{f_Y(Y)} \cdot g(Y)]\;.
$$

From this fact, we may, instead, estimate $E_X[g(X)]$ using $n$ draws $y_1, \ldots, y_n$ from $f_Y$ and use the sample mean
$$
\frac{1}{n} \sum_{i=1}^n \frac{f_X(y_i)}{f_Y(y_i)} \cdot g(y_i)\;. 
$$

So even if we define $f_Y$ to better align with $g$ which artificially generates "rarer" events more frequently compared to the original distribution, the transformation ratio we see above: $\frac{f_X(x)}{f_Y(x)}$, is applied to each draw. In regions where the event is rare under $f_X$ but common under $f_Y$, this ratio is a small fraction that mathematically scales down the massive $g(x)$ values, ensuring the final expected value is not artificially inflated by the skewed sampling.

### Coding Up Importance Sampling

In [127]:



def importance_sampling(target, proposal, sampler, g, n_samples):
    # Unpack
    t_pdf, t_args, t_kwargs = target
    p_pdf, p_args, p_kwargs = proposal
    p_samp, p_samp_args, p_samp_kwargs = sampler

    # draw n_samples from proposal density
    p_samp_kwargs["size"] = n_samples
    p_samples = p_samp(*p_samp_args, **p_samp_kwargs)

    # Approximate via average
    f_X = t_pdf(p_samples, *t_args, **t_kwargs)
    f_Y = p_pdf(p_samples, *p_args, **p_kwargs)
    importance_ratio = f_X / f_Y # n_samples sized array 
    elements = importance_ratio * g(p_samples)
    E_approx = (1/n_samples) * np.sum(elements)

    return E_approx

**Test Case**

Target distribution: Normal(0,1).

Function $g(x) = \mathbb{I}(x > 4.5)$, indicator function returning 1 if $x > 4.5$ and $0$ otherwise.

The integral to estimate is $E_X[g(X)] = P(X > 4.5)$.

First let us attempt to approximate this expectation using standard Monte Carlo integration. That is, we will draw $N = 10,000$ samples from $N(0,1)$, and then compute the sample average. What we can expect is most if not all samples will not exceed $4.5$ leading to a approximation of $0$ for the expectation.

In [ ]:
# draw samples
N = int(1e4)
x_samples = stats.norm.rvs(size=N)

# function to compute
def g_indicator(x):
    if (x > 4.5):
        return 1
    else:
        return 0

# vectorize our function
vec_g_indicator = np.vectorize(g_indicator)

# compute average
elements = vec_g_indicator(x_samples)
MC_avg = (1/N) * np.sum(elements)

print(f"Approximation of the expectation: {MC_avg}")

# Computing the true analytical answer
Ex = stats.norm.sf(4.5)
print(f"The true analytical answer is {Ex}")

Approximation of the expectation: 0.0
The true analytical answer is 3.3976731247300543e-06


Now let us use importance sampling to see if we can get a better approximation. We will propose a distribution that sits over the region where $g(x)$ is active. We propose a normal distribution with mean 4.5 and variance of 1 to shift the mass over the threshold:

$$
f_Y(y) = \frac{1}{\sqrt{2\pi}} e^{-\frac{(y-4.5)^2}{2}}\;.
$$

Now we proceed with the algorithm:

In [134]:
target = (stats.norm.pdf, (), {})
proposal = (stats.norm.pdf, (), {"loc" : 4.5, "scale" : 1})
sampler = (stats.norm.rvs, (), {"loc" : 4.5, "scale" : 1})

# function to compute
def g_indicator(x):
    if (x > 4.5):
        return 1
    else:
        return 0

# vectorize our function
vec_g_indicator = np.vectorize(g_indicator)

IS_avg = importance_sampling(target, proposal, sampler, vec_g_indicator, n_samples=int(1e4))

print(f"Approximation of the expectation: {IS_avg}")

# Computing the true analytical answer
Ex = stats.norm.sf(4.5)
print(f"The true analytical answer is {Ex}")


Approximation of the expectation: 3.3423682811629204e-06
The true analytical answer is 3.3976731247300543e-06


**Comparing the Variances of the Approximation Error of the Above Tests**

The variance of the approximation error is the variance of the sample mean in either case:
$$
Var(E_X[g(X)] - \hat\mu_{method}) = Var(\hat\mu_{method})\;.
$$

**MC integration**
$$
V_{MC} = \frac{1}{N} Var(g(X))
$$

**IS**
$$
V_{IS} = \frac{1}{N} Var\left(g(Y) \cdot \frac{f_X(Y)}{f_Y(Y)}\right)
$$

In [135]:
# We will use log pdfs for numerical stability
p_samples = stats.norm.rvs(loc=4.5, scale=1.0, size=10000)
g_p_samples = vec_g_indicator(p_samples)

# Weights
log_f_X = stats.norm.logpdf(p_samples)
log_f_Y = stats.norm.logpdf(p_samples, loc=4.5, scale=1.0)
imp_weights = np.exp(log_f_X - log_f_Y)

# variance of approximation error for IS
Z = g_p_samples * imp_weights
approx_err_var_IS = (1/10000) * np.var(Z, ddof=1)

# variance of approximation error for MC
approx_err_var_MC = (1/10000) * np.var(elements, ddof=1)

print(f"Variance of approximation error using Importance Sampling: {approx_err_var_IS}")
print(f"Variance of approximation error using Monte Carlo Integration: {approx_err_var_MC}")


Variance of approximation error using Importance Sampling: 5.962823991267323e-15
Variance of approximation error using Monte Carlo Integration: 0.0


While it looks like MC integration has no approximation error, this is actually a trap. Standard Monte Carlo reports a zero variance not because the estimate is flawless, but because it lacks the sample size required to see the rare event, trapping the estimator in a sample space entirely made of zeros.